# Submissão B – Implementação com PyTorch  
**Grupo 7 | MIA | Aprendizagem Profunda**  
**Modelo:** Embeddings treináveis + Bidirectional LSTM + classificador DNN  
> ⚠️ Requer `torch` instalado. Sem transformers.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import re
import pickle
from collections import Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
torch.manual_seed(42)
np.random.seed(42)


Device: cpu
PyTorch: 2.10.0


In [2]:
CLASSES   = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
LABEL2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2LABEL = {i: c for i, c in enumerate(CLASSES)}

TRAIN_PATH = 'dataset-exemplos.csv'
TEST_PATH  = 'subm1.csv'
OUT_PATH   = 'subm1-g7-MIA-B.csv'
MODEL_PATH = 'model_B.pt'


## 1. Carregamento e Pré-processamento


In [3]:
train_df = pd.read_csv(TRAIN_PATH, sep=';')
test_df  = pd.read_csv(TEST_PATH,  sep=';', encoding='utf-8-sig')

print(f'Treino: {len(train_df)} | Teste: {len(test_df)}')
print(train_df['Label'].value_counts())


Treino: 125 | Teste: 150
Label
Human        52
Anthropic    23
Meta         17
OpenAI       17
Google       16
Name: count, dtype: int64


In [4]:
STOPWORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'by','from','is','are','was','were','be','been','being','have','has',
    'had','do','does','did','will','would','could','should','may','might',
    'shall','can','this','that','these','those','it','its','they','them',
    'their','we','our','you','your','he','she','his','her','i','my','me',
])

def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 1]

# Construir vocabulário
PAD_TOKEN, UNK_TOKEN = '<PAD>', '<UNK>'
MAX_VOCAB  = 5000
MAX_SEQ    = 100  # máx. tokens por exemplo

all_tokens = []
for text in train_df['Text']:
    all_tokens.extend(tokenize(text))

token_freq = Counter(all_tokens)
vocab_words = [PAD_TOKEN, UNK_TOKEN] + [w for w, _ in token_freq.most_common(MAX_VOCAB)]
word2idx = {w: i for i, w in enumerate(vocab_words)}

print(f'Vocabulário: {len(word2idx)} tokens')


Vocabulário: 3080 tokens


In [5]:
def text_to_ids(text, max_len=MAX_SEQ):
    """Converte texto para sequência de IDs com padding/truncation."""
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(t, word2idx[UNK_TOKEN]) for t in tokens]
    # Padding à direita
    ids += [word2idx[PAD_TOKEN]] * (max_len - len(ids))
    return ids

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.X = [text_to_ids(t) for t in texts]
        self.y = labels

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx], dtype=torch.long)
        if self.y is not None:
            return x, torch.tensor(self.y[idx], dtype=torch.long)
        return x

# Preparar datasets
texts_train = train_df['Text'].tolist()
y_train     = [LABEL2IDX[l] for l in train_df['Label']]
texts_test  = test_df['Text'].tolist()

# Split 80/20
N = len(texts_train)
perm  = np.random.permutation(N)
split = int(0.8 * N)
tr_idx, val_idx = perm[:split], perm[split:]

ds_train = TextDataset([texts_train[i] for i in tr_idx], [y_train[i] for i in tr_idx])
ds_val   = TextDataset([texts_train[i] for i in val_idx], [y_train[i] for i in val_idx])
ds_test  = TextDataset(texts_test)

BATCH = 16
dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH)
dl_test  = DataLoader(ds_test,  batch_size=BATCH)

print(f'Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}')


Train: 100 | Val: 25 | Test: 150


## 2. Arquitectura do Modelo: Embeddings + BiLSTM + Classifier


In [6]:
class TextClassifier(nn.Module):
    """
    Arquitectura:
      1. Embedding layer (treináveis, dimensão EMB_DIM)
      2. Bidirectional LSTM com dropout
      3. Pooling (max + mean) sobre a sequência
      4. Classificador MLP com dropout
    """
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=128,
                 n_layers=2, n_classes=5, dropout=0.4, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size    = emb_dim,
            hidden_size   = hidden_dim,
            num_layers    = n_layers,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout if n_layers > 1 else 0.0,
        )

        # Após BiLSTM: hidden_dim * 2 (bidirecional) * 2 (max + mean pool)
        lstm_out_dim = hidden_dim * 2 * 2

        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)           # (batch, seq_len, emb_dim)
        out, _ = self.lstm(emb)           # (batch, seq_len, hidden*2)
        # Pooling
        max_pool  = out.max(dim=1).values # (batch, hidden*2)
        mean_pool = out.mean(dim=1)       # (batch, hidden*2)
        pooled = torch.cat([max_pool, mean_pool], dim=1)  # (batch, hidden*4)
        return self.classifier(pooled)    # (batch, n_classes)


model = TextClassifier(
    vocab_size = len(word2idx),
    emb_dim    = 64,
    hidden_dim = 128,
    n_layers   = 2,
    n_classes  = len(CLASSES),
    dropout    = 0.4,
    pad_idx    = word2idx[PAD_TOKEN],
).to(DEVICE)

print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParâmetros treináveis: {n_params:,}')


TextClassifier(
  (embedding): Embedding(3080, 64, padding_idx=0)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (classifier): Sequential(
    (0): Linear(in_features=512, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.4, inplace=False)
    (6): Linear(in_features=64, out_features=5, bias=True)
  )
)

Parâmetros treináveis: 865,285


## 3. Treino


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

N_EPOCHS  = 100
PATIENCE  = 20

best_val_loss = float('inf')
wait = 0
best_state = None

for epoch in range(N_EPOCHS):
    # ── Treino ──
    model.train()
    train_losses = []
    for X_batch, y_batch in dl_train:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        train_losses.append(loss.item())

    # ── Validação ──
    model.eval()
    val_losses, val_preds, val_true = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in dl_val:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            val_losses.append(criterion(logits, y_batch).item())
            val_preds.extend(logits.argmax(dim=1).cpu().numpy())
            val_true.extend(y_batch.cpu().numpy())

    tl = np.mean(train_losses)
    vl = np.mean(val_losses)
    va = np.mean(np.array(val_preds) == np.array(val_true))
    scheduler.step(vl)

    if vl < best_val_loss:
        best_val_loss = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f'Early stopping @ epoch {epoch+1}')
            break

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:4d}  train={tl:.4f}  val={vl:.4f}  acc={va:.3f}')

# Restaurar melhores pesos
model.load_state_dict(best_state)
print('\nTreino concluído. Melhores pesos restaurados.')


Epoch   10  train=1.0846  val=2.0219  acc=0.240
Epoch   20  train=0.9055  val=2.4625  acc=0.080
Early stopping @ epoch 22

Treino concluído. Melhores pesos restaurados.


## 4. Re-treino no Conjunto Completo e Previsão


In [8]:
# Re-treinar no conjunto completo
ds_full  = TextDataset(texts_train, y_train)
dl_full  = DataLoader(ds_full, batch_size=BATCH, shuffle=True)

model_full = TextClassifier(
    vocab_size = len(word2idx),
    emb_dim    = 64,
    hidden_dim = 128,
    n_layers   = 2,
    n_classes  = len(CLASSES),
    dropout    = 0.3,
    pad_idx    = word2idx[PAD_TOKEN],
).to(DEVICE)

opt2 = optim.Adam(model_full.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(60):
    model_full.train()
    for X_batch, y_batch in dl_full:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        opt2.zero_grad()
        loss = criterion(model_full(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model_full.parameters(), 1.0)
        opt2.step()

print('Re-treino no conjunto completo concluído.')


Re-treino no conjunto completo concluído.


In [9]:
# Previsões no conjunto de teste
model_full.eval()
all_preds = []
with torch.no_grad():
    for X_batch in dl_test:
        X_batch = X_batch.to(DEVICE)
        logits  = model_full(X_batch)
        preds   = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)

labels = [IDX2LABEL[i] for i in all_preds]
print('Distribuição das previsões:')
print(Counter(labels))

# Guardar modelo
torch.save({
    'model_state_dict': model_full.state_dict(),
    'word2idx': word2idx,
    'classes':  CLASSES,
    'config':   {'vocab_size': len(word2idx), 'emb_dim': 64,
                 'hidden_dim': 128, 'n_layers': 2,
                 'n_classes': len(CLASSES), 'dropout': 0.3}
}, MODEL_PATH)
print(f'Modelo guardado: {MODEL_PATH}')

# Gerar CSV
out_df = test_df[['ID', 'Text']].copy()
out_df['Label'] = labels
out_df.to_csv(OUT_PATH, sep=';', index=False)
print(f'CSV gerado: {OUT_PATH}')
out_df.head(10)


Distribuição das previsões:
Counter({'Google': 60, 'Human': 51, 'Anthropic': 21, 'OpenAI': 15, 'Meta': 3})
Modelo guardado: model_B.pt
CSV gerado: subm1-g7-MIA-B.csv


,ID,Text,Label
0,D2-1,A covalent bond is a chemical bond that involv...,Google
1,D2-2,A covalent bond forms when two atoms share one...,Anthropic
2,D2-3,A covalent bond is a type of chemical bond whe...,Google
3,D2-4,A covalent bond is a chemical bond that involv...,Google
4,D2-5,Driven by exciting developments in the field o...,Google
5,D2-6,Ionic bonding results from the electrostatic a...,Google
6,D2-7,Plate tectonics is the scientific theory expla...,Google
7,D2-8,Plate tectonics is a fundamental scientific th...,Google
8,D2-9,Tectonic plates are relatively rigid and float...,Google
9,D2-10,"At present, young oceanic lithosphere is posit...",Google
